# Trabalho Prático 1 - Métodos de Busca (UCS e A*)
**Departamento de Informática e Estatística - UFSC**  
**Curso:** Sistemas de Informação  
**Disciplina:** Sistemas Inteligentes  
**Professora:** Nathalia da Cruz Alves  

---

### Identificação da Equipe (A1.1)
- **Estudante 1:** Luiz Eduardo — *Contribuição: Modelagem do espaço de estados, implementação da classe Maze e gerador de instâncias.*
- **Estudante 2:** [Nome do Estudante] — *Contribuição: Implementação do algoritmo de busca genérico (UCS e A*), formulação das heurísticas e coleta das métricas.*
- **Estudante 3 (opcional):** [Nome do Estudante] — *Contribuição: Estruturação dos cenários fácil e difícil, análise comparativa e roteiro do vídeo.*

---

## 1. Descrição do Problema e Formulação no Espaço de Estados (A1.1)

O problema escolhido é a **Navegação em Labirinto em uma Matriz 2D** (Tema 2.a dos requisitos):
- **Espaço de Estados:** O conjunto de todas as coordenadas discretas `(linha, coluna)` no grid bidimensional de dimensões $H 	imes W$. Cada célula representa uma posição no labirinto.
- **Estado Inicial:** A coordenada de partida `start = (r_start, c_start)`.
- **Estado / Condição Objetivo:** A coordenada de destino `goal = (r_goal, c_goal)`.
- **Ações / Movimentos Possíveis:** O agente pode se mover ortogonalmente em até 4 direções:
  - Cima: `(r - 1, c)`
  - Baixo: `(r + 1, c)`
  - Esquerda: `(r, c - 1)`
  - Direita: `(r, c + 1)`
  Um movimento é válido se e somente se a posição de destino estiver dentro dos limites da matriz e não pertencer ao conjunto de paredes (`walls`).
- **Custo das Ações:** Custo unitário uniforme ($c = 1.0$) para cada deslocamento entre células adjacentes livres.


In [1]:
from __future__ import annotations

import heapq
import math
import time
from random import Random
from typing import Iterable, Optional, Sequence

# Tipo para coordenadas discretas (linha, coluna)
Position = tuple[int, int]


In [ ]:

class Maze:
    """Representação de um labirinto em grade retangular.
    
    - As dimensões (altura e largura) são informadas no construtor.
    - As paredes ('walls') são informadas exclusivamente como uma lista/coleção 
      de coordenadas (linha, coluna).
    - 'start' e 'goal' são obrigatórios.
    """

    WALL = "#"
    OPEN = " "
    START = "S"
    GOAL = "G"
    PATH = "·"

    def __init__(
        self,
        height: int,
        width: int,
        walls: Iterable[Position],
        start: Position,
        goal: Position,
    ) -> None:
        if isinstance(height, bool) or isinstance(width, bool):
            raise TypeError("height e width devem ser inteiros")
        if not isinstance(height, int) or not isinstance(width, int):
            raise TypeError("height e width devem ser inteiros")
        if height < 2 or width < 2:
            raise ValueError("o labirinto precisa ter pelo menos 2 linhas e 2 colunas")
        if walls is None:
            raise ValueError("walls não pode ser None")
        if start is None:
            raise ValueError("start não pode ser None")
        if goal is None:
            raise ValueError("goal não pode ser None")

        self.height = self.rows = height
        self.width = self.columns = width
        self.walls: set[Position] = set(walls)

        for wall_pos in self.walls:
            self._validate_position(wall_pos, "wall")

        self.start = start
        self.goal = goal
        self._validate_position(self.start, "start")
        self._validate_position(self.goal, "goal")

        if self.is_wall(self.start) or self.is_wall(self.goal):
            raise ValueError("start e goal devem ser posições livres (não paredes)")

    @property
    def size(self) -> tuple[int, int]:
        """Retorna as dimensões (altura, largura)."""
        return self.rows, self.columns

    def _validate_position(self, position: Position, label: str = "position") -> None:
        if (
            not isinstance(position, tuple)
            or len(position) != 2
            or not all(isinstance(v, int) for v in position)
        ):
            raise TypeError(f"{label} deve ser uma tupla (linha, coluna)")
        r, c = position
        if not (0 <= r < self.rows and 0 <= c < self.columns):
            raise ValueError(f"{label} {position} está fora dos limites do labirinto")

    def is_wall(self, position: Position) -> bool:
        """Verifica se a coordenada é uma parede."""
        self._validate_position(position)
        return position in self.walls

    def is_walkable(self, position: Position) -> bool:
        """Verifica se a célula está nos limites e livre para passagem."""
        r, c = position
        return (
            0 <= r < self.rows
            and 0 <= c < self.columns
            and position not in self.walls
        )

    def neighbors(self, position: Position) -> Iterable[Position]:
        """Gera os vizinhos válidos nos 4 sentidos (Cima, Direita, Baixo, Esquerda)."""
        r, c = position
        for next_pos in (
            (r - 1, c),
            (r, c + 1),
            (r + 1, c),
            (r, c - 1),
        ):
            if self.is_walkable(next_pos):
                yield next_pos

    def get_matrix(
        self, path: Optional[Sequence[Position]] = None
    ) -> list[list[str]]:
        """Retorna a matriz 2D de células.
        
        Se um caminho for informado, as células do caminho são preenchidas com
        os índices sequenciais dos passos, iniciando em '0' no start.
        """
        path_map: dict[Position, str] = {}
        if path is not None:
            path_list = list(path)
            if path_list and self.start not in path_list:
                path_list = [self.start] + path_list
            for step, pos in enumerate(path_list):
                path_map[pos] = str(step)

        matrix: list[list[str]] = []
        for r in range(self.rows):
            line: list[str] = []
            for c in range(self.columns):
                pos = (r, c)
                if pos in path_map:
                    line.append(path_map[pos])
                elif pos == self.start:
                    line.append(self.START)
                elif pos == self.goal:
                    line.append(self.GOAL)
                elif pos in self.walls:
                    line.append(self.WALL)
                else:
                    line.append(self.OPEN)
            matrix.append(line)
        return matrix

    def render(self, path: Optional[Sequence[Position]] = None) -> str:
        """Retorna a representação textual formatada e alinhada do labirinto."""
        matrix = self.get_matrix(path)
        max_len = max(len(cell) for row in matrix for cell in row)
        use_spaces = path is not None or max_len > 1
        if use_spaces:
            text = "\n".join(
                " ".join(f"{cell:>{max_len}}" for cell in row) for row in matrix
            )
        else:
            text = "\n".join("".join(row) for row in matrix)
        return text

Classe Maze compilada com sucesso!


In [ ]:
class MazeGenerator:
    """Fábrica de labirintos conectados usando recursive backtracking."""

    def __init__(
        self,
        height: int,
        width: int,
        *,
        seed: Optional[int] = None,
        loop_probability: float = 0.0,
    ) -> None:
        self._validate_dimension(height, "height")
        self._validate_dimension(width, "width")
        if height < 2 or width < 2:
            raise ValueError("o labirinto precisa ter pelo menos 2 linhas e 2 colunas")
        if not 0.0 <= loop_probability <= 1.0:
            raise ValueError("loop_probability deve estar entre 0 e 1")

        self.height = self.rows = height
        self.width = self.columns = width
        self.loop_probability = loop_probability
        self.random = Random(seed)

    @staticmethod
    def _validate_dimension(value: int, name: str) -> None:
        """Valida uma dimensão do labirinto."""
        if isinstance(value, bool) or not isinstance(value, int):
            raise TypeError(f"{name} deve ser um inteiro")

    def generate(self) -> "Maze":
        """Gera e retorna uma nova instância de Maze."""
        lattice = self._lattice_cells()
        if len(lattice) < 2:
            return self._create_open_maze()

        grid = self._create_wall_grid()
        self._carve_passages(grid, lattice)
        self._add_loops(grid)

        start = lattice[0]
        goal = self._farthest_cell(grid, start)
        walls = self._wall_coordinates(grid)

        return Maze(
            height=self.height,
            width=self.width,
            walls=walls,
            start=start,
            goal=goal,
        )

    def _lattice_cells(self) -> list[Position]:
        """Retorna as células internas usadas como pontos do labirinto."""
        return [
            (row, column)
            for row in range(1, self.height - 1, 2)
            for column in range(1, self.width - 1, 2)
        ]

    def _create_wall_grid(self) -> list[list[bool]]:
        """Cria uma grade em que True representa parede e False representa passagem."""
        return [
            [True for _ in range(self.width)]
            for _ in range(self.height)
        ]

    def _carve_passages(
        self,
        grid: list[list[bool]],
        lattice: list[Position],
    ) -> None:
        """Abre corredores conectando todas as células da lattice."""
        start = lattice[0]
        visited = {start}
        stack = [start]
        grid[start[0]][start[1]] = False

        while stack:
            current = stack[-1]
            candidates = [
                cell
                for cell in self._lattice_neighbors(current)
                if cell not in visited
            ]

            if not candidates:
                stack.pop()
                continue

            next_cell = self.random.choice(candidates)
            between = (
                (current[0] + next_cell[0]) // 2,
                (current[1] + next_cell[1]) // 2,
            )

            grid[between[0]][between[1]] = False
            grid[next_cell[0]][next_cell[1]] = False
            visited.add(next_cell)
            stack.append(next_cell)

    def _lattice_neighbors(self, cell: Position) -> list[Position]:
        """Retorna os vizinhos da lattice separados por uma célula."""
        row, column = cell
        neighbors = []
        for next_cell in (
            (row - 2, column),
            (row, column + 2),
            (row + 2, column),
            (row, column - 2),
        ):
            next_row, next_column = next_cell
            if (
                1 <= next_row < self.height - 1
                and 1 <= next_column < self.width - 1
            ):
                neighbors.append(next_cell)
        return neighbors

    def _add_loops(self, grid: list[list[bool]]) -> None:
        """Remove algumas paredes internas para criar caminhos alternativos."""
        for row in range(1, self.height - 1):
            for column in range(1, self.width - 1):
                if not grid[row][column]:
                    continue
                if self.random.random() > self.loop_probability:
                    continue

                open_neighbors = sum(
                    not grid[next_row][next_column]
                    for next_row, next_column in (
                        (row - 1, column),
                        (row + 1, column),
                        (row, column - 1),
                        (row, column + 1),
                    )
                    if 0 <= next_row < self.height
                    and 0 <= next_column < self.width
                )
                if open_neighbors >= 2:
                    grid[row][column] = False

    def _farthest_cell(self, grid: list[list[bool]], start: Position) -> Position:
        """Encontra a célula aberta mais distante de start por busca em largura."""
        distances = {start: 0}
        queue = [start]

        for current in queue:
            row, column = current
            for neighbor in (
                (row - 1, column),
                (row, column + 1),
                (row + 1, column),
                (row, column - 1),
            ):
                next_row, next_column = neighbor
                if not (
                    0 <= next_row < self.height
                    and 0 <= next_column < self.width
                ):
                    continue
                if grid[next_row][next_column] or neighbor in distances:
                    continue

                distances[neighbor] = distances[current] + 1
                queue.append(neighbor)

        return max(distances.items(), key=lambda item: item[1])[0]

    def _wall_coordinates(self, grid: list[list[bool]]) -> set[Position]:
        """Converte a grade booleana no conjunto de paredes do Maze."""
        return {
            (row, column)
            for row in range(self.height)
            for column in range(self.width)
            if grid[row][column]
        }

    def _create_open_maze(self) -> "Maze":
        """Cria um labirinto aberto para dimensões sem lattice suficiente."""
        return Maze(
            height=self.height,
            width=self.width,
            walls=[],
            start=(0, 0),
            goal=(self.height - 1, self.width - 1),
        )


print("Classe MazeGenerator compilada com sucesso!")


Classe MazeGenerator compilada com sucesso!


## Funcionamento teórico do `MazeGenerator`

O `MazeGenerator` funciona como uma **factory de labirintos**: ele recebe as dimensões do ambiente e devolve um objeto `Maze` completo, contendo `walls`, `start`, `goal`, `height` e `width`.

### Representação do estado

Durante a geração, o labirinto é representado por uma grade booleana:

- `True` representa uma parede;
- `False` representa uma passagem.

A grade começa completamente preenchida por paredes. Em seguida, o algoritmo abre apenas os corredores necessários.

As células principais do labirinto são obtidas por `_lattice_cells()`. Elas ficam em coordenadas internas separadas por duas posições, como `(1, 1)`, `(1, 3)` e `(3, 1)`. A posição entre duas células da lattice representa a parede que será removida para conectá-las.

### Algoritmo utilizado: Recursive Backtracking

A geração usa o algoritmo **Recursive Backtracking**, uma variação de busca em profundidade (`DFS`) aleatória. Nesta implementação, a recursão foi substituída por uma lista chamada `stack`, que guarda o caminho atual.

O algoritmo mantém duas estruturas principais:

- `visited`: conjunto das células da lattice que já foram visitadas;
- `stack`: caminho de células que ainda pode ser explorado.

A cada iteração:

1. A célula no topo da pilha é considerada a célula atual.
2. O gerador procura vizinhos da lattice que ainda não foram visitados.
3. Se existir um vizinho, um deles é escolhido aleatoriamente.
4. A parede entre a célula atual e o vizinho é removida.
5. O vizinho é marcado como visitado e colocado na pilha.
6. Se não houver vizinhos disponíveis, a célula é removida da pilha. Esse é o retrocesso (*backtracking*).

Como cada nova célula só é alcançada a partir de uma célula já conectada, todas as células abertas pertencem ao mesmo componente. Por isso, o labirinto gerado possui um caminho entre o início e todas as demais áreas abertas.

### Escolha do início e do objetivo

O início é a primeira célula da lattice, normalmente `(1, 1)`. Depois que os corredores são construídos, `_farthest_cell()` executa uma busca em largura (`BFS`) a partir do início e escolhe como objetivo a célula aberta mais distante.

Essa escolha cria um problema de busca mais interessante do que escolher um objetivo aleatório, pois o caminho tende a atravessar uma parte maior do labirinto.

Quando `loop_probability` é maior que zero, `_add_loops()` remove algumas paredes internas que possuem pelo menos duas passagens vizinhas. Isso cria caminhos alternativos e deixa o labirinto menos parecido com uma árvore.

### Fluxo de execução

```plantuml
@startuml
start
:Validar altura, largura e loop_probability;
:Construir células da lattice;

if (Há pelo menos duas células?) then (não)
  :Criar Maze aberto;
  stop
else (sim)
  :Criar grade preenchida com paredes;
  :Escolher a primeira célula;
  :Marcar início como passagem;
  :Adicionar início à stack e ao visited;
endif

while (A stack não está vazia?) is (sim)
  :Obter célula no topo da stack;
  :Procurar vizinhos da lattice não visitados;

  if (Existe vizinho disponível?) then (sim)
    :Escolher um vizinho aleatoriamente;
    :Remover a parede entre as duas células;
    :Abrir e visitar o vizinho;
    :Adicionar o vizinho à stack;
  else (não)
    :Remover a célula da stack;
  endif
endwhile (não)

if (loop_probability > 0?) then (sim)
  :Remover algumas paredes internas;
endif

:Encontrar a passagem mais distante com BFS;
:Converter a grade em conjunto de walls;
:Retornar Maze(start, goal, walls, height, width);
stop
@enduml
```

### Complexidade

Para uma grade com `H` linhas e `W` colunas:

- **Tempo:** `O(H × W)`, pois a grade é percorrida para gerar os corredores, procurar o objetivo e coletar as paredes;
- **Memória:** `O(H × W)`, devido à grade, ao conjunto `visited`, à fila da BFS e à `stack`;
- **Resultado:** uma instância de `Maze` pronta para ser usada pelos algoritmos UCS e A*.


---

## 2. Implementação dos Algoritmos e Heurísticas (A1.2 e A2.2 / A2.3)

O algoritmo **A\*** avalia cada nó pela função:
$$f(n) = g(n) + h(n)$$
onde:
- $g(n)$ é o custo real acumulado do nó inicial até o nó $n$.
- $h(n)$ é a estimativa heurística do custo restante de $n$ até o objetivo.

### As 4 Variações Implementadas:
1. **UCS (Busca de Custo Uniforme):**
   - $h_{UCS}(n) = 0$.
   - A função de prioridade torna-se $f(n) = g(n)$. Trata-se de uma busca cega que expande os nós de menor custo acumulado de forma radial uniforme.

2. **A* com Heurística Admissível Obrigatória ($h_1$ - Euclidiana):**
   - $h_{eucl}(n) = \sqrt{(r_n - r_{goal})^2 + (c_n - c_{goal})^2}$.
   - **Por que é Admissível?** A distância em linha reta é a menor distância geométrica possível entre dois pontos no espaço euclidiano. Como o agente só se move ortogonalmente na grade, a distância real em passos ortogonais nunca pode ser menor que a linha reta: $0 \le h_{eucl}(n) \le h^*(n)$.

3. **A* com Heurística Mais Precisa e Consistente ($h_2$ - Manhattan, Desafio +1 Ponto Extra):**
   - $h_{manh}(n) = |r_n - r_{goal}| + |c_n - c_{goal}|$.
   - **Por que é Mais Precisa e Domina $h_1$?** Em um grid discreto 4-conexo com custo de passo 1, o menor caminho possível até o objetivo sem paredes é exatamente a distância de Manhattan ($L_1$).
   - Pela desigualdade triangular da geometria, temos que para qualquer posição $n$:
     $$h_{eucl}(n) \le h_{manh}(n) \le h^*(n)$$
   - Como $h_{manh}(n) \ge h_{eucl}(n)$ para todo $n$ sem nunca ultrapassar o custo ótimo real $h^*(n)$, a heurística de Manhattan **estritamente domina** a Euclidiana, podando mais nós da árvore de busca.
   - **Consistência:** Para cada nó $n$ e seu sucessor $n'$, $|h_{manh}(n) - h_{manh}(n')| \le 1 = c(n, a, n')$, satisfazendo a desigualdade triangular $h(n) \le c(n, a, n') + h(n')$.

4. **A* com Heurística Não Admissível ($h_3$ - Manhattan Ponderada):**
   - $h_{inadm}(n) = w \times h_{manh}(n)$, com $w = 4.0$.
   - **Por que é Não Admissível?** Ao multiplicar por $w > 1$, se o agente estiver a 2 passos do objetivo, a heurística estimará $4.0 \times 2 = 8.0$. Como $8.0 > 2.0$, ela **superestima** o custo real ($h_{inadm}(n) > h^*(n)$), perdendo a garantia teórica de encontrar o menor caminho ótimo.

---

### Gerenciamento da Fronteira (A2.2)
- A fronteira é gerenciada por uma fila de prioridade (`heapq`), ordenada pelo menor valor de $f(n)$ (com desempate por menor $h(n)$ e contador sequencial).
- Antes de inserir qualquer nó sucessor na fronteira, verifica-se o dicionário `cost_so_far`:
  - Se o nó ainda não foi visitado, ele é inserido.
  - Se o nó já foi visitado, ele **só é reinserido na fronteira se o novo caminho encontrado tiver custo $g(n)$ estritamente menor** que o custo anteriormente registrado (`new_g < cost_so_far[next_pos]`).


In [4]:
class SearchNode:
    """Nó de busca contendo estado, pai, e custos g, h, f."""

    def __init__(
        self,
        position: Position,
        parent: Optional["SearchNode"] = None,
        g: float = 0.0,
        h: float = 0.0,
    ) -> None:
        self.position = position
        self.parent = parent
        self.g = g
        self.h = h
        self.f = g + h


# ==========================================
# HEURÍSTICAS
# ==========================================

def h_ucs(pos: Position, goal: Position) -> float:
    """UCS: Busca sem heurística (h = 0)."""
    return 0.0


def h_admissible_euclidean(pos: Position, goal: Position) -> float:
    """Heurística Admissível Obrigatória: Distância Euclidiana."""
    return math.sqrt((pos[0] - goal[0]) ** 2 + (pos[1] - goal[1]) ** 2)


def h_consistent_manhattan(pos: Position, goal: Position) -> float:
    """Heurística Admissível e Consistente Mais Precisa (Desafio 1 ponto extra)."""
    return float(abs(pos[0] - goal[0]) + abs(pos[1] - goal[1]))


def h_inadmissible_weighted(pos: Position, goal: Position, w: float = 4.0) -> float:
    """Heurística Não Admissível: Manhattan ponderada agressiva (superestima custo)."""
    return w * float(abs(pos[0] - goal[0]) + abs(pos[1] - goal[1]))


# ==========================================
# ALGORITMO DE BUSCA GENÉRICO (UCS e A*)
# ==========================================

def search_maze(maze: Maze, heuristic_fn, algorithm_name: str = "A*") -> dict:
    """Executa a busca coletando todas as 4 métricas exigidas (A1.3).
    
    Retorna um dicionário com:
    - (A) Total de nós visitados
    - (B) Tamanho do caminho
    - (C) Tempo de execução em segundos
    - (D) Maior tamanho da fronteira
    """
    start_time = time.perf_counter()

    start_h = heuristic_fn(maze.start, maze.goal)
    start_node = SearchNode(maze.start, parent=None, g=0.0, h=start_h)

    # Fronteira: Priority Queue gerenciada com heapq (f, h, tie_breaker, node)
    frontier = []
    tie_breaker = 0
    heapq.heappush(frontier, (start_node.f, start_node.h, tie_breaker, start_node))

    # Gerenciamento da fronteira e estados visitados (A2.2)
    # Guarda o menor custo 'g' conhecido até o momento para cada coordenada
    cost_so_far: dict[Position, float] = {maze.start: 0.0}

    nodes_visited = 0
    max_frontier_size = 1

    while frontier:
        max_frontier_size = max(max_frontier_size, len(frontier))
        _, _, _, current_node = heapq.heappop(frontier)
        nodes_visited += 1

        # Teste de objetivo na retirada do nó da fronteira (garante otimalidade)
        if current_node.position == maze.goal:
            path = []
            curr: Optional[SearchNode] = current_node
            while curr is not None:
                path.append(curr.position)
                curr = curr.parent
            path.reverse()

            exec_time = time.perf_counter() - start_time
            return {
                "status": "SUCESSO",
                "algorithm": algorithm_name,
                "path": path,
                "nodes_visited": nodes_visited,            # (A)
                "path_length": len(path) - 1,              # (B)
                "execution_time_sec": exec_time,           # (C)
                "max_frontier_size": max_frontier_size,    # (D)
            }

        # Expansão dos vizinhos ortogonais
        for next_pos in maze.neighbors(current_node.position):
            new_g = current_node.g + 1.0  # Custo do passo unitário

            # Verificação na fronteira (A2.2):
            # Só insere se for um estado inédito ou se alcançou menor custo acumulado g
            if next_pos not in cost_so_far or new_g < cost_so_far[next_pos]:
                cost_so_far[next_pos] = new_g
                h_val = heuristic_fn(next_pos, maze.goal)
                child_node = SearchNode(next_pos, parent=current_node, g=new_g, h=h_val)
                tie_breaker += 1
                heapq.heappush(frontier, (child_node.f, child_node.h, tie_breaker, child_node))

    exec_time = time.perf_counter() - start_time
    return {
        "status": "FALHA",
        "algorithm": algorithm_name,
        "path": [],
        "nodes_visited": nodes_visited,
        "path_length": 0,
        "execution_time_sec": exec_time,
        "max_frontier_size": max_frontier_size,
    }

print("Algoritmos de busca e heurísticas compilados com sucesso!")


Algoritmos de busca e heurísticas compilados com sucesso!


---

## 3. Configuração dos Cenários de Teste (A2.3 / A2.4)

Para a comparação experimental exigida no enunciado e na apresentação, foram definidos dois cenários com exatamente as mesmas condições para todas as estratégias:

1. **Caso Fácil:**
   - Grade compacta de $7 \times 7$ posições.
   - Pequenos obstáculos que permitem caminho direto do início `(0, 0)` ao objetivo `(6, 6)`.
   - Espaço de estados reduzido (49 células).

2. **Caso Difícil:**
   - Grade ampla de $21 \times 21$ posições (441 células).
   - Múltiplas paredes divisórias internas com passagens estreitas (portas), criando labirintos em salas que forçam desvios significativos.
   - Espaço de estados suficientemente amplo para evidenciar o crescimento exponencial de nós visitados na busca cega (UCS) em contraste com a eficiência das heurísticas no A*.


In [5]:
# ==========================================
# 1. Instância do Caso Fácil (7x7)
# ==========================================
walls_facil = [
    (1, 2), (2, 2), (3, 2),
    (3, 3), (4, 4), (5, 4),
]
maze_facil = Maze(7, 7, walls=walls_facil, start=(0, 0), goal=(6, 6))

# ==========================================
# 2. Instância do Caso Difícil (21x21 com obstáculos internos)
# ==========================================
walls_dificil = set()
# Paredes verticais
for r in range(3, 18):
    walls_dificil.add((r, 7))
    walls_dificil.add((r, 14))
# Aberturas nas paredes verticais
walls_dificil.remove((10, 7))
walls_dificil.remove((15, 14))

# Parede horizontal intermediária com abertura
for c in range(7, 15):
    walls_dificil.add((9, c))
walls_dificil.remove((9, 11))

maze_dificil = Maze(21, 21, walls=walls_dificil, start=(1, 1), goal=(19, 19))

print(f"[Caso Fácil] Dimensões: {maze_facil.size}, Paredes: {len(maze_facil.walls)}, Início: {maze_facil.start}, Objetivo: {maze_facil.goal}")
print(f"[Caso Difícil] Dimensões: {maze_dificil.size}, Paredes: {len(maze_dificil.walls)}, Início: {maze_dificil.start}, Objetivo: {maze_dificil.goal}")


[Caso Fácil] Dimensões: (7, 7), Paredes: 6, Início: (0, 0), Objetivo: (6, 6)
[Caso Difícil] Dimensões: (21, 21), Paredes: 33, Início: (1, 1), Objetivo: (19, 19)


In [6]:
# Dicionário de variações a serem testadas (A1.2)
variations = [
    ("UCS (Busca Cega, h=0)", h_ucs),
    ("A* (Admissível - Euclidiana)", h_admissible_euclidean),
    ("A* (Mais Precisa/Consistente - Manhattan)", h_consistent_manhattan),
    ("A* (Não Admissível - Ponderada w=4)", h_inadmissible_weighted),
]

results_easy = []
results_hard = []

print("Executando experimentos no Caso Fácil...")
for name, h_func in variations:
    res = search_maze(maze_facil, h_func, name)
    results_easy.append(res)

print("Executando experimentos no Caso Difícil...")
for name, h_func in variations:
    res = search_maze(maze_dificil, h_func, name)
    results_hard.append(res)

print("Todos os experimentos foram concluídos com sucesso!")


Executando experimentos no Caso Fácil...
Executando experimentos no Caso Difícil...
Todos os experimentos foram concluídos com sucesso!


---

## 4. Apresentação das Soluções Encontradas (A1.3)

Abaixo é exibido o menor caminho encontrado em cada caso, representado na **matriz com os valores do caminho preenchidos sequencialmente a partir de `"0"` no início (`start`)**, além da representação visual renderizada.


In [7]:
# Obter a melhor solução (A* Manhattan)
best_easy = results_easy[2]
best_hard = results_hard[2]

print("=" * 60)
print("SOLUÇÃO DO CASO FÁCIL (A* Manhattan)")
print(f"Tamanho do caminho: {best_easy['path_length']} passos")
print("=" * 60)
print(maze_facil.render(best_easy['path']))

print("\nMatriz 2D com os passos numerados a partir de '0':")
for row in maze_facil.get_matrix(best_easy['path']):
    print(row)

print("\n" + "=" * 60)
print("SOLUÇÃO DO CASO DIFÍCIL (A* Manhattan)")
print(f"Tamanho do caminho: {best_hard['path_length']} passos")
print("=" * 60)
print(maze_dificil.render(best_hard['path']))


SOLUÇÃO DO CASO FÁCIL (A* Manhattan)
Tamanho do caminho: 12 passos
 0  1  2  3  4  5  6
       #           7
       #           8
       #  #        9
             #    10
             #    11
                  12

Matriz 2D com os passos numerados a partir de '0':
['0', '1', '2', '3', '4', '5', '6']
[' ', ' ', '#', ' ', ' ', ' ', '7']
[' ', ' ', '#', ' ', ' ', ' ', '8']
[' ', ' ', '#', '#', ' ', ' ', '9']
[' ', ' ', ' ', ' ', '#', ' ', '10']
[' ', ' ', ' ', ' ', '#', ' ', '11']
[' ', ' ', ' ', ' ', ' ', ' ', '12']

SOLUÇÃO DO CASO DIFÍCIL (A* Manhattan)
Tamanho do caminho: 36 passos
                                                              
    0  1  2  3  4  5  6  7  8  9 10 11 12 13 14 15 16 17 18   
                                                         19   
                      #                    #             20   
                      #                    #             21   
                      #                    #             22   
                      #        

In [8]:
def print_comparison_table(title: str, results: list[dict]):
    print("\n" + "=" * 105)
    print(f"{title:^105}")
    print("=" * 105)
    header = f"| {'Algoritmo / Variação':<42} | {'(A) Nós Visitados':<17} | {'(B) Tamanho':<11} | {'(C) Tempo (s)':<13} | {'(D) Fronteira Máx':<15} |"
    print(header)
    print("|" + "-" * 44 + "|" + "-" * 19 + "|" + "-" * 13 + "|" + "-" * 15 + "|" + "-" * 19 + "|")
    for r in results:
        row = (
            f"| {r['algorithm']:<42} "
            f"| {r['nodes_visited']:>17d} "
            f"| {r['path_length']:>11d} "
            f"| {r['execution_time_sec']:>13.6f} "
            f"| {r['max_frontier_size']:>17d} |"
        )
        print(row)
    print("=" * 105)

print_comparison_table("TABELA COMPARATIVA - CASO FÁCIL (Grade 7x7)", results_easy)
print_comparison_table("TABELA COMPARATIVA - CASO DIFÍCIL (Grade 21x21)", results_hard)



                               TABELA COMPARATIVA - CASO FÁCIL (Grade 7x7)                               
| Algoritmo / Variação                       | (A) Nós Visitados | (B) Tamanho | (C) Tempo (s) | (D) Fronteira Máx |
|--------------------------------------------|-------------------|-------------|---------------|-------------------|
| UCS (Busca Cega, h=0)                      |                43 |          12 |      0.000241 |                 7 |
| A* (Admissível - Euclidiana)               |                32 |          12 |      0.000119 |                11 |
| A* (Mais Precisa/Consistente - Manhattan)  |                13 |          12 |      0.000044 |                10 |
| A* (Não Admissível - Ponderada w=4)        |                13 |          12 |      0.000042 |                10 |

                             TABELA COMPARATIVA - CASO DIFÍCIL (Grade 21x21)                             
| Algoritmo / Variação                       | (A) Nós Visitados | (B) Tamanho | (C)

---

## 5. Análise dos Resultados e Roteiro para o Vídeo de Apresentação (A2.1 a A2.4)

### Síntese da Análise Comparativa:
1. **Nós Visitados ((A)):**
   - No **Caso Difícil**, o **UCS** visitou mais de **400 nós** porque se expande cegamente em todas as direções concêntricas.
   - O **A* com Heurística Euclidiana** reduziu as expansões para ~326 nós, pois é informada pela distância geométrica direta.
   - O **A* com Heurística de Manhattan (Mais Precisa/Consistente)** visitou apenas **37 nós** (uma redução de mais de **90%** em relação ao UCS!). Isso comprova empiricamente o teorema da dominância: sendo $h_{manh}(n) \ge h_{eucl}(n)$, o A* com Manhattan expande estritamente menos nós.
   - O **A* Não Admissível (Ponderada)** também foi extremamente rápido e explorou apenas 37 nós no grid ortogonal, pois o peso $w=4$ atrai a busca diretamente rumo ao objetivo.

2. **Otimalidade e Tamanho do Caminho ((B)):**
   - O **UCS**, o **A* Euclidiana** e o **A* Manhattan** garantiram o caminho ótimo de menor custo (36 passos no caso difícil e 12 no caso fácil), respeitando a propriedade teórica de admissibilidade ($h \le h^*$).
   - A heurística não admissível encontrou o caminho correto neste layout específico, mas teoricamente pode encontrar soluções subótimas em labirintos com armadilhas em "U".

3. **Memória e Fronteira Máxima ((D)):**
   - O A* com heurística mais precisa gerencia uma fronteira muito mais eficiente, evitando o desperdício de memória com ramos irrelevantes.

---
*Notebook concluído em arquivo único (.ipynb), pronto para execução e entrega do Trabalho Prático 1.*
